# Assignment 6 Project-Aligned LoRA Run

Runs the frozen candidate on the approved Development-only split. The official Held-out Evaluation data are not included.


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import zipfile

home = Path('/mnt/nas/students/yl14754')
bundle = home / 'a6_project_route_lora_bundle.zip'
with zipfile.ZipFile(bundle) as archive:
    for member in archive.namelist():
        target = (home / member).resolve()
        if not target.is_relative_to(home.resolve()):
            raise ValueError(f'Unsafe archive member: {member}')
    archive.extractall(home)

job_script = home / 'a6_project_route_lora_job.sh'
job_script.write_text(
    '#!/bin/bash\n'
    'set -euo pipefail\n'
    'export PYTHONPATH=/mnt/nas/students/yl14754/a6_python_packages\n'
    '/opt/sps-ai-lab/venv/bin/python /mnt/nas/students/yl14754/project_route_lora_train.py '
    '--config /mnt/nas/students/yl14754/project_route_lora_config.json\n',
    encoding='utf-8',
)
job_script.chmod(0o755)
log_pattern = str(home / 'a6_project_route_lora_%j.log')
command = [
    'sbatch', '--wait', '--parsable', '--partition=ai-lab', '--nodelist=spark-4694',
    '--gres=gpu:gb10:1', '--cpus-per-task=4', '--mem=24G', '--time=00:20:00',
    f'--output={log_pattern}', str(job_script),
]
submission = subprocess.run(command, capture_output=True, text=True, timeout=1500)
print('SBATCH_RETURN', submission.returncode)
print('SBATCH_STDOUT', submission.stdout.strip())
print('SBATCH_STDERR', submission.stderr.strip())
if submission.returncode != 0:
    raise RuntimeError('Slurm rejected the Assignment 6 project-data LoRA job')
job_id = submission.stdout.strip().split(';')[0]
log_path = Path(log_pattern.replace('%j', job_id))
print('SLURM_JOB_ID', job_id)
print('LOG_PATH', log_path)
if not log_path.exists():
    raise FileNotFoundError(f'Slurm output log not found: {log_path}')
print('--- JOB LOG ---')
print(log_path.read_text(encoding='utf-8', errors='replace')[-24000:])
result_path = home / 'a6_project_route_result.json'
if result_path.exists():
    print('--- RESULT JSON ---')
    print(json.dumps(json.loads(result_path.read_text(encoding='utf-8')), indent=2))
